# TrafficFlowBench: density post-processing for Task 3

**Why.** The official Task 3 scorer (`src/task3/score_task3.py`) derives density from the Task 1 answer,
`N = (q / v) · L`, and scores conservation `S_LWR = 1 − Σ|ΔN − rhs| / Σ|rhs|` on 5-minute transitions. The organizer
boundary flows are *projected onto the released observations*, so the true field satisfies `ΔN = rhs` exactly. A
transition therefore loses points only through the error of `N` at predicted (hidden) cells. The released
neighbours in time pin `N` down tightly: it changes by well under 1% per step. Interpolating `N` between them may be
far more accurate than `q̂ / v̂` from the state models.

**What this notebook does.**
1. On February 2031 (held out from the public CatBoost state models), hide cells at 20/30/50%, predict them with
   those models, and compute exactly the official `S_state`, the `S_FD` formula, and the `S_LWR` identity proxy.
2. Grid over a correction that moves `q̂ / v̂` towards the interpolated density: strength `w`, split `α` between flow (α = 1) and
   speed (α = 0), and maximum hidden run length.
3. Apply the best setting to the 0.87141 submission **only if** the estimated combined gain
   `0.35·ΔS_state + 0.15·(ΔS_FD/3 + 2·ΔS_LWR/3)` exceeds `min_gain`.

**Inputs:** competition data · notebook output `anubisyy/catboost-lgbm-nnls-pipeline-top-50-0-86517` (`tfb_source/`) ·
dataset `anubisyy/trafficflowbench-086517-public-assets` (state checkpoints) · output holding the 0.87141 `submission.csv`.
**Accelerator: GPU** (CatBoost inference). CPU also works but is slower.

Credit: Thanks to the work of [Prabhav Semwal](https://www.kaggle.com/code/yeashusemwal/trafficflowbench-trees-u-net-cnn-lb-0-87).

In [ ]:
import os, sys, glob, shutil, time, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

CFG = dict(
    base_sub=None,      # path of the 0.87141 submission.csv; None = auto-detect
    days=7,             # February days per panel used for the local evaluation
    min_gain=0.001,     # required estimated LB gain before the submission is changed
    force=None,         # e.g. dict(w=1.0, alpha=1.0, max_run=2) to skip auto-selection
    grid_w=(0.25, 0.5, 0.75, 1.0), grid_alpha=(0.0, 0.25, 0.5, 0.75, 1.0), grid_run=(1, 2, 4, 999),
)
T0 = time.time(); elapsed = lambda: f"{time.time() - T0:.0f}s"
found = lambda pat: sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True))
DATA = Path(found("kaggle_public")[0])
SRC_IN = os.path.dirname([p for p in found("state_inference.py")
                          if all(os.path.exists(os.path.join(os.path.dirname(p), f)) for f in ("state_model.py", "prepare_state.py", "base_panel.py", "config.py"))][0])
ASSETS = os.path.dirname(found("speed_d8.cbm")[0])
subs = found("submission.csv")
BASE_SUB = CFG["base_sub"] or ([p for p in subs if "catboost" not in p] or subs)[0]
print("data:", DATA); print("source:", SRC_IN); print("checkpoints:", ASSETS); print("BASE submission:", BASE_SUB)

WORK = Path("/kaggle/working"); SRC = WORK / "tfb_src"
shutil.copytree(SRC_IN, SRC, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
os.environ.update(TFB_DATA_ROOT=str(DATA), TFB_OUTPUT_ROOT=str(WORK / "tfb_out"), TFB_THREADS="4")
sys.path.insert(0, str(SRC))
import state_model as sm
from prepare_state import features
from state_inference import StatePredictor
predictor = StatePredictor(ASSETS)
PANELS = sm.all_panels()
print("panels:", PANELS, elapsed())

## Shared pieces

`interp_N` uses only released (eligible, finite) cells as anchors, exactly what the validation and private files
show. `adjust` moves the predicted density towards the interpolated one in log space:
`q ← q̂·e^{α·w·δ}`, `v ← v̂·e^{−(1−α)·w·δ}`, with `δ = log k_interp − log(q̂/v̂)`.

In [ ]:
def link_geometry(p, name):
    net = DATA / "corridors" / name / "network"
    links = pd.read_csv(net / "links.csv"); links["link_id"] = links.link_id.astype(str)
    L = pd.to_numeric(links.drop_duplicates("link_id").set_index("link_id").length_km, errors="coerce").reindex(p.links).to_numpy(float)
    L = np.where(np.isfinite(L), L, 1.0)
    fd = pd.read_csv(net / "fd_parameters.csv"); fd["link_id"] = fd.link_id.astype(str)
    have = [c for c in ("free_speed_kmh", "capacity_vph", "lanes", "k_jam") if c in fd]
    m = fd.groupby("link_id")[have].mean().reindex(p.links)
    lanes = m.lanes.fillna(1.0).clip(lower=1.0).to_numpy(float)
    vf = m.free_speed_kmh.to_numpy(float); cap = m.capacity_vph.to_numpy(float)
    kcrit = cap / np.maximum(vf, 1)
    kjam = (m.k_jam if "k_jam" in m else pd.Series(np.nan, index=m.index)).to_numpy(float)
    kjam = np.where(np.isfinite(kjam), kjam, 100.0 * lanes); kjam = np.maximum(kjam, 1.05 * kcrit)
    return dict(L=L, lanes=lanes, vf=vf, cap=cap, kcrit=kcrit, kjam=kjam)

def interp_N(v_obs, q_obs, e, L, rows, cols):
    ok = e & np.isfinite(v_obs) & np.isfinite(q_obs)
    N = np.where(ok, q_obs / np.maximum(v_obs, 1.0) * L[None, :], np.nan)
    T = N.shape[0]; pos = np.arange(T)[:, None]
    prev = np.maximum.accumulate(np.where(ok, pos, -1), axis=0)
    nxt = np.minimum.accumulate(np.where(ok, pos, T)[::-1], axis=0)[::-1]
    p, n = prev[rows, cols], nxt[rows, cols]
    has = (p >= 0) & (n < T); pp, nn = np.maximum(p, 0), np.minimum(n, T - 1)
    t = (rows - pp) / np.maximum(nn - pp, 1)
    Ni = N[pp, cols] * (1 - t) + N[nn, cols] * t
    return np.where(has, Ni, np.nan), np.where(has, nn - pp - 1, 10**6)

def adjust(v_hat, q_hat, Ni, run, L_cells, cfg):
    k_hat = q_hat / np.maximum(v_hat, 1.0); k_i = Ni / L_cells
    ok = np.isfinite(k_i) & (k_i > 0) & (k_hat > 0) & (run <= cfg["max_run"])
    d = np.where(ok, np.log(np.maximum(k_i, 1e-9)) - np.log(np.maximum(k_hat, 1e-9)), 0.0) * cfg["w"]
    return np.clip(v_hat * np.exp(-(1 - cfg["alpha"]) * d), 1, 180), np.clip(q_hat * np.exp(cfg["alpha"] * d), 0, 150000)

def scores(P, vn, qn):
    """Official S_state, S_FD formula and the S_LWR identity proxy, per regime present."""
    g, rows, cols = P["geo"], P["rows"], P["cols"]
    V, Q = P["v"].copy(), P["q"].copy(); V[rows, cols] = vn; Q[rows, cols] = qn
    N = Q / np.maximum(V, 1.0) * g["L"][None, :]; Nt = P["Nt"]
    dN, dNt = N[1:] - N[:-1], Nt[1:] - Nt[:-1]
    lanes = g["lanes"][None, :]
    ql, kl = Q / lanes, Q / np.maximum(V, 1.0) / lanes
    kc, kj, cp = (g["kcrit"] / g["lanes"])[None, :], (g["kjam"] / g["lanes"])[None, :], (g["cap"] / g["lanes"])[None, :]
    qfd = np.where(kl <= kc, g["vf"][None, :] * kl, cp / np.maximum(kj - kc, 1e-9) * np.maximum(kj - kl, 0))
    out = {}
    for r in np.unique(P["reg_cell"]):
        c = P["reg_cell"] == r
        rv = np.sqrt(np.mean((vn[c] - P["vt"][c]) ** 2)); rq = np.sqrt(np.mean(((qn[c] - P["qt"][c]) / g["lanes"][cols[c]]) ** 2))
        s_state = 0.54 * max(0, 1 - rv / 25) + 0.46 * max(0, 1 - rq / 600)
        tr = P["trans"] & (P["reg_row"][:-1] == r)[:, None]
        s_lwr = max(0.0, 1 - np.abs(dN - dNt)[tr].sum() / max(np.abs(dNt)[tr].sum(), 1e-9))
        u = P["used"] & (P["reg_row"] == r)[:, None]
        s_fd = max(0.0, 1 - np.abs(ql - qfd)[u].sum() / max(np.abs(ql)[u].sum(), 1e-9))
        out[int(r)] = dict(S_state=s_state, S_LWR=s_lwr, S_FD=s_fd, rmse_v=rv, rmse_q_lane=rq)
    return out

## 1. February held-out: predict hidden cells with the public CatBoost state models

In [ ]:
PROBS = []
for pi, name in enumerate(PANELS):
    p = sm.Panel(name, pi)
    fr = sm.month_frame(name, "2031-02")
    fr["timestamp"] = fr.timestamp.astype(str)
    days = sorted(fr.timestamp.str[:10].unique())[:CFG["days"]]
    fr = fr[fr.timestamp.str[:10].isin(days)]
    times, v, q, e = sm.matrices(fr, p)
    mask, regime = sm.simulated_mask(name, times, v, q, e)
    rows, cols = np.nonzero(mask)
    vo, qo = np.where(mask, np.nan, v), np.where(mask, np.nan, q)
    X, bv, bq = features(p, pi, times, vo, qo, e, rows, cols, regime[rows])
    vh = np.clip(bv + predictor.residual(X, "speed"), 1, 180)
    qh = np.clip((bq + predictor.residual(X, "flow")) * p.lanes[cols], 0, 150000)
    geo = link_geometry(p, name)
    used = e & np.isfinite(v) & np.isfinite(q)
    ts = pd.to_datetime(pd.Index(times), utc=True)
    step = np.asarray((ts[1:] - ts[:-1]).total_seconds() == 300)
    Ni, run = interp_N(vo, qo, e, geo["L"], rows, cols)
    PROBS.append(dict(name=name, geo=geo, rows=rows, cols=cols, v=np.where(used, v, np.nan), q=np.where(used, q, np.nan),
                      vt=v[rows, cols], qt=q[rows, cols], vh=vh, qh=qh, Ni=Ni, run=run, used=used,
                      trans=used[:-1] & used[1:] & step[:, None], reg_row=regime, reg_cell=regime[rows],
                      Nt=np.where(used, q / np.maximum(v, 1.0), np.nan) * geo["L"][None, :]))
    kt = q[rows, cols] / np.maximum(v[rows, cols], 1); kh = qh / np.maximum(vh, 1); ki = Ni / geo["L"][cols]
    f = np.isfinite(ki)
    print(f"{name}: {len(days)} days, hidden {len(rows):,}, density MAE model {np.mean(np.abs(kh - kt)):.3f} "
          f"vs interpolation {np.mean(np.abs(ki[f] - kt[f])):.3f} (on {f.mean():.0%} with anchors)", elapsed(), flush=True)

## 2. Grid over the correction

In [ ]:
def evaluate(cfg):
    rows = []
    for P in PROBS:
        vn, qn = (P["vh"], P["qh"]) if cfg is None else adjust(P["vh"], P["qh"], P["Ni"], P["run"], P["geo"]["L"][P["cols"]], cfg)
        for r, s in scores(P, vn, qn).items():
            rows.append(dict(panel=P["name"], regime=r, **s))
    d = pd.DataFrame(rows)
    return d, d.drop(columns="regime").groupby("panel").mean(numeric_only=True).mean()

base_detail, base = evaluate(None)
print("baseline (CatBoost only, macro mean):"); print(base.round(4).to_string())
print(base_detail.groupby("regime")[["S_state", "S_LWR", "S_FD"]].mean().round(4).to_string())
res = []
for w in CFG["grid_w"]:
    for a in CFG["grid_alpha"]:
        for mr in CFG["grid_run"]:
            cfg = dict(w=w, alpha=a, max_run=mr); _, m = evaluate(cfg)
            d = m - base
            res.append(dict(**cfg, dS_state=d.S_state, dS_LWR=d.S_LWR, dS_FD=d.S_FD,
                            est_LB=0.35 * d.S_state + 0.15 * (d.S_FD / 3 + 2 * d.S_LWR / 3)))
res = pd.DataFrame(res).sort_values("est_LB", ascending=False).reset_index(drop=True)
res.to_csv(WORK / "t3_grid.csv", index=False)
print(res.head(15).round(5).to_string(), elapsed())

In [ ]:
best = res.iloc[0]
if CFG["force"]:
    CHOSEN, why = dict(CFG["force"]), "forced"
elif best.est_LB >= CFG["min_gain"]:
    CHOSEN, why = dict(w=float(best.w), alpha=float(best.alpha), max_run=int(best.max_run)), f"est. LB {best.est_LB:+.4f}"
else:
    CHOSEN, why = None, f"best est. LB {best.est_LB:+.4f} below {CFG['min_gain']}"
print("chosen:", CHOSEN, "|", why)
if CHOSEN:
    det, _ = evaluate(CHOSEN)
    cmp = base_detail.set_index(["panel", "regime"])[["S_state", "S_LWR", "S_FD"]].join(
        det.set_index(["panel", "regime"])[["S_state", "S_LWR", "S_FD"]], rsuffix="_new")
    print(cmp.round(4).to_string())

## 3. Apply to the 0.87141 submission

Anchors come from the released masked files of each split. The predicted speed and flow are the base submission's
values (the 50/50 blend), so only the correction is new.

In [ ]:
sub = pd.read_csv(BASE_SUB)
key = pd.read_csv(DATA / "submission_key.csv", dtype=str, usecols=["submission_id", "task", "panel", "timestamp", "link_id"])
if pd.api.types.is_numeric_dtype(sub.submission_id):
    key["submission_id"] = pd.to_numeric(key.submission_id)
key = key.set_index("submission_id").reindex(sub.submission_id.values).reset_index()
assert key.task.notna().all(), "base submission ids missing from submission_key"
ks = key[key.task == "state"]
if CHOSEN:
    changed = 0
    for pi, name in enumerate(PANELS):
        p = sm.Panel(name, pi); geo = link_geometry(p, name); kp = ks[ks.panel == name]
        for split in ("validation", "private"):
            fr = sm.month_frame(name, "", split=split); fr["timestamp"] = fr.timestamp.astype(str)
            times, v, q, e = sm.matrices(fr, p)
            r = pd.Index(times).get_indexer(kp.timestamp); sel = r >= 0
            rows, idx = r[sel], kp.index.values[sel]
            cols = pd.Index(p.links).get_indexer(kp.link_id.values[sel]); assert (cols >= 0).all()
            Ni, run = interp_N(v, q, e, geo["L"], rows, cols)
            vh, qh = sub.speed_kmh.values[idx].astype(float), sub.flow_vph.values[idx].astype(float)
            vn, qn = adjust(vh, qh, Ni, run, geo["L"][cols], CHOSEN)
            changed += int(((np.abs(vn - vh) > 1e-6) | (np.abs(qn - qh) > 1e-6)).sum())
            sub.loc[idx, "speed_kmh"] = vn; sub.loc[idx, "flow_vph"] = qn
            print(name, split, f"{sel.sum():,} cells, anchored {np.isfinite(Ni).mean():.0%}", elapsed(), flush=True)
    print(f"state cells changed: {changed:,}")

tpl = pd.read_csv(DATA / "sample_submission.csv", usecols=["submission_id"])
st = sub.task == "state"
checks = {"row count": len(sub) == len(tpl), "template order": bool((sub.submission_id.values == tpl.submission_id.values).all()),
          "no missing": not sub.isna().any().any(), "all finite": bool(np.isfinite(sub[["speed_kmh", "flow_vph", "path_flow"]].to_numpy()).all()),
          "binary queues": set(sub.queue_pred.unique()) <= {0, 1}, "empty-road guard": float((sub.flow_vph[st] < 50).mean()) <= 0.2}
assert all(checks.values()), checks
sub.to_csv(WORK / "submission.csv", index=False)
print("written /kaggle/working/submission.csv", elapsed()); print(pd.Series(checks).to_string())